In [1]:
   %pip install python-dotenv

   import os
   from dotenv import load_dotenv

   load_dotenv()
   api = os.getenv("GROQ_API_KEY")

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import requests

In [3]:
url = "https://api.groq.com/openai/v1/models"

In [4]:
from crewai import Agent, Task, Process, LLM, Crew

In [5]:
llm=LLM(
    model="groq/openai/gpt-oss-20b",
    api_key=api,
    temperature=0.3, #creativity
    timeout=120, #timeout and max_tokens to improve efficiency
    max_tokens=3000 #max num of words or sentence to gen
)

In [6]:
travel_planner = Agent(
    role="Professional Travel Planner",

    goal="""
    Create personalized, practical, and enjoyable travel itineraries
    based on the user's destination, travel dates, trip duration,
    number of travelers, interests, and budget.

    Recommend tourist attractions, accommodation, transportation,
    restaurants, and activities.

    Organize the trip into a clear day-wise itinerary, considering
    travel time, location, convenience, and the user's preferences.

    Coordinate with the Budget Advisor to ensure the proposed
    itinerary is financially practical and within the user's budget.
    """,

    backstory="""
    You are an experienced travel consultant who specializes in
    creating personalized and budget-friendly travel experiences.

    You understand:
    - Travel destination research
    - Tourist attractions and sightseeing
    - Day-wise itinerary planning
    - Transportation options
    - Accommodation recommendations
    - Local experiences and activities
    - Restaurant and food recommendations
    - Travel time and distance considerations
    - Seasonal travel considerations
    - Budget-conscious travel planning

    You always understand the user's travel requirements before
    creating an itinerary.

    You ask concise questions when essential information is missing.

    You prioritize enjoyable, practical, and well-organized travel plans.

    You avoid unnecessary travel expenses and recommend suitable
    alternatives when needed.

    You never invent hotel prices, transportation schedules,
    attraction opening hours, or travel information.

    You clearly distinguish verified information, estimates,
    assumptions, and recommendations.

    You collaborate with the Budget Advisor to make sure the
    itinerary aligns with the user's financial limitations.
    """,

    llm=llm,

    verbose=True,

    allow_delegation=False,

    max_iter=2
)

In [7]:
budget_advisor = Agent(
    role="Professional Travel Budget Advisor",

    goal="""
    Analyse the travel itinerary and estimate the total cost
    of the trip based on the user's specified budget.

    Calculate estimated expenses for transportation, accommodation,
    food, sightseeing, activities, and miscellaneous expenses.

    Provide a detailed category-wise budget breakdown and compare
    the estimated total cost with the user's available budget.

    Identify expensive activities and recommend affordable
    alternatives and practical money-saving strategies.

    Ensure that the final travel plan stays within the user's
    budget without unnecessarily compromising the travel experience.
    """,

    backstory="""
    You are an experienced travel financial consultant specializing
    in travel expense estimation, budget management, and cost optimization.

    You understand:
    - Travel expense estimation
    - Transportation cost calculations
    - Hotel and accommodation pricing
    - Food and dining expenses
    - Tourist attraction and activity costs
    - Daily travel budget planning
    - Currency and expense calculations
    - Budget allocation
    - Cost comparison
    - Money-saving strategies
    - Emergency and miscellaneous expenses

    You always analyse the user's available budget and travel
    requirements before estimating expenses.

    You calculate expenses carefully and provide transparent
    category-wise cost breakdowns.

    You recommend affordable transportation, accommodation,
    dining, and activity alternatives whenever necessary.

    You never invent exact prices or present estimates as confirmed costs.

    You clearly distinguish estimated costs from verified prices.

    You never ignore hidden or additional expenses when preparing
    the budget.

    If the estimated trip cost exceeds the user's budget, you
    communicate with the Travel Planner and recommend necessary
    adjustments to the itinerary.

    Your priority is to help users enjoy their trips while
    maintaining financial control.
    """,

    llm=llm,

    verbose=True,

    allow_delegation=False,

    max_iter=2
)

In [8]:
travel_task = Task(
    description="""
    Create a personalized travel itinerary based on the user's request:

    {user_request}

    Follow these instructions:

    1. Understand the user's destination, travel dates, and trip duration.
    2. Identify the number of travelers.
    3. Understand the user's travel interests and preferences.
    4. Consider the user's total available budget.
    5. Ask concise questions if essential information is missing.

    Create a detailed travel itinerary that includes:

    - Destination overview
    - Recommended tourist attractions
    - Day-wise travel itinerary
    - Activities and sightseeing options
    - Transportation recommendations
    - Accommodation suggestions
    - Food and dining recommendations
    - Estimated time required for each activity
    - Travel tips and important considerations

    Additional instructions:

    - Organize the itinerary day by day.
    - Consider travel distances and time between attractions.
    - Avoid unnecessary travel and backtracking.
    - Recommend suitable attractions based on the user's interests.
    - Prioritize affordable and convenient travel options.
    - Do not exceed the user's budget intentionally.
    - Share the complete itinerary with the Budget Advisor
      for expense estimation.

    Never invent hotel availability, exact prices, transportation
    schedules, or attraction opening hours.

    Clearly distinguish verified information, assumptions,
    and recommendations.

    Make the itinerary practical, enjoyable, and easy to follow.
    """,

    expected_output="""
    A concise travel itinerary containing:

    1. Destination overview
    2. Day-wise itinerary
    3. Attractions and activities
    4. Transportation suggestions
    5. Accommodation suggestions
    6. Estimated expenses for each activity

    Keep the output clear and practical.
    Avoid unnecessary explanations.
    """,

    agent=travel_planner
)

In [9]:
budget_task = Task(
    description="""
    Analyse the travel itinerary created by the Travel Planner
    and prepare a detailed estimated travel budget.

    User's original request:

    {user_request}

    Travel itinerary:
    Use the itinerary provided by the Travel Planner as context.

    Follow these instructions:

    1. Understand the user's total available budget.
    2. Analyse the proposed travel itinerary.
    3. Estimate transportation expenses.
    4. Estimate accommodation expenses.
    5. Calculate food and dining expenses.
    6. Estimate sightseeing and activity expenses.
    7. Include miscellaneous and emergency expenses.
    8. Calculate the total estimated trip cost.
    9. Compare the total cost with the user's budget.

    Provide a detailed budget breakdown including:

    - Transportation cost
    - Accommodation cost
    - Food and dining cost
    - Sightseeing and activity cost
    - Miscellaneous expenses
    - Emergency allowance
    - Total estimated budget
    - Remaining budget or additional amount required

    If the estimated trip cost exceeds the user's budget:

    - Identify the most expensive components.
    - Suggest cheaper accommodation options.
    - Recommend affordable transportation.
    - Suggest free or low-cost attractions.
    - Recommend practical ways to reduce expenses.
    - Suggest itinerary adjustments to the Travel Planner.

    Additional instructions:

    - Show calculations clearly.
    - Consider the number of travelers and trip duration.
    - Avoid double-counting expenses.
    - Clearly mention when prices are estimates.
    - Never invent exact prices or confirmed booking availability.
    - Do not ignore additional expenses.
    - Keep recommendations realistic and practical.

    Your priority is to ensure that the final travel plan
    stays within the user's available budget.
    """,

    expected_output="""
    A concise travel budget report containing:
    
    1. Transportation cost
    2. Accommodation cost
    3. Food expenses
    4. Activities and sightseeing costs
    5. Miscellaneous expenses
    6. Emergency allowance
    7. Total estimated cost
    8. Remaining budget
    9. Money-saving recommendations
    
    Compare the total estimated cost with the user's budget.
    
    If the budget is exceeded, suggest affordable alternatives.
    Show calculations clearly and avoid unnecessary explanations.
    """,

    agent=budget_advisor,

    context=[travel_task]
)

In [10]:
travel_crew = Crew(
    agents=[travel_planner,budget_advisor],
    tasks=[travel_task,budget_task],
    process=Process.sequential,
    verbose=True
)

In [11]:
user_request = """
Plan a budget-friendly 3-day trip to Goa for 2 people.

Destination: Goa, India
Duration: 3 days and 2 nights
Number of travelers: 2
Total Budget: Rs. 15,000

Travel Preferences:
- Visit beautiful beaches and popular tourist attractions.
- Explore local markets and famous sightseeing locations.
- Enjoy local Goan food.
- Include affordable adventure activities.
- Prefer peaceful and scenic places.

Accommodation:
- Suggest affordable hotels or budget-friendly stays.
- Prefer accommodation near major tourist attractions.

Transportation:
- Recommend economical transportation options.
- Consider travel time and distance between destinations.

Food:
- Suggest affordable restaurants and local food options.

Special Requirements:
- Keep the total trip cost within Rs. 15,000.
- Provide a day-wise travel itinerary.
- Include transportation, accommodation, food,
  sightseeing, and miscellaneous expenses.
- Suggest money-saving tips and affordable alternatives.

Final Output:
Provide a complete 3-day travel itinerary,
a detailed category-wise estimated budget,
total trip cost, remaining budget, and
money-saving recommendations.
"""

In [12]:
result = await travel_crew.kickoff_async(
    inputs={
        "user_request": user_request
    }
)

print("\n" + "=" * 60)
print("BUDGET-FRIENDLY TRAVEL PLANNING RESPONSE")
print("=" * 60)

print(result)

# Agent: Professional Travel Planner
## Task: 
    Create a personalized travel itinerary based on the user's request:

    
Plan a budget-friendly 3-day trip to Goa for 2 people.

Destination: Goa, India
Duration: 3 days and 2 nights
Number of travelers: 2
Total Budget: Rs. 15,000

Travel Preferences:
- Visit beautiful beaches and popular tourist attractions.
- Explore local markets and famous sightseeing locations.
- Enjoy local Goan food.
- Include affordable adventure activities.
- Prefer peaceful and scenic places.

Accommodation:
- Suggest affordable hotels or budget-friendly stays.
- Prefer accommodation near major tourist attractions.

Transportation:
- Recommend economical transportation options.
- Consider travel time and distance between destinations.

Food:
- Suggest affordable restaurants and local food options.

Special Requirements:
- Keep the total trip cost within Rs. 15,000.
- Provide a day-wise travel itinerary.
- Include transportation, accommodation, food,
  sights

Transient error HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by NameResolutionError("HTTPSConnection(host='telemetry.crewai.com', port=4319): Failed to resolve 'telemetry.crewai.com' ([Errno 11002] getaddrinfo failed)")) encountered while exporting spans batch, retrying in 1.06s.
Transient error HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by NameResolutionError("HTTPSConnection(host='telemetry.crewai.com', port=4319): Failed to resolve 'telemetry.crewai.com' ([Errno 11002] getaddrinfo failed)")) encountered while exporting spans batch, retrying in 2.34s.
Transient error HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by NameResolutionError("HTTPSConnection(host='telemetry.crewai.com', port=4319): Failed to resolve 'telemetry.crewai.com' ([Errno 11002] getaddrinfo failed)")) encountered while e



# Agent: Professional Travel Planner
## Final Answer: 
**

---

## 1. Destination Overview  
Goa, the smallest state of India, is famed for its golden‑sanded beaches, Portuguese heritage, vibrant markets, and mouth‑watering seafood. The 3‑day itinerary focuses on the popular North‑Goa coastline (Baga, Calangute, Anjuna) and a touch of South‑Goa’s serene beaches (Palolem, Agonda), while keeping the budget tight.

---

## 2. Day‑wise Itinerary  

| Day | Morning | Afternoon | Evening | Notes |
|-----|---------|-----------|---------|-------|
| **Day 1** | Arrive in Goa (by train/coach). Take local bus to **Calangute**. Check‑in at budget homestay. | Explore **Calangute Beach**; walk to **Baga** for a quick snack. | Sunset at **Calangute**; light dinner at a beach shack. | Total travel time from airport/railway: ~1 h. |
| **Day 2** | Breakfast at homestay. Take shared auto to **Anjuna Market** (open 10 am‑6 pm). | Visit **Anjuna Beach**; optional low‑cost water‑sport (kayak/stand‑up padd